# Experiment 5 — Residual Late Fusion (Baseline-Preserving) — Colab Run

`z_final = z_rgb + delta_z`: `z_rgb` from the **frozen, trained RGB MiT-B1+CORN baseline** (loaded per fold). `delta_z` from `TissueOnlyResidualHead` — **tissue-only variant** (default, `--variant tissue_only`): the residual head sees ONLY tissue CNN features, never RGB, so delta_z structurally cannot be a function of RGB content. GroupNorm (not BatchNorm) in the tissue CNN, dropout=0.2 in the residual head. Final layer zero-initialized as always: at construction z_final == z_rgb exactly.

**Why tissue-only, not the original design:** the first version (`--variant rgb_tissue`, residual head saw RGB+tissue concatenated) was diagnosed (`diagnose_residual.py`) to produce a correction nearly independent of tissue content (cosine(real,shuffled tissue) ~1.0) — plausibly because RGB gave the optimizer an easier path to reduce loss than actually using tissue. Fold 1's local run of `tissue_only` improved on this (monotonic val-F1 improvement instead of peaking at epoch 1, final result matching/slightly beating the baseline) but the SAME tissue-ablation diagnostic still shows near-zero content-dependence (relative L2(real,shuffled)/||delta_z|| ~ 0.005, tissue CNN gradient norm ~0.001) -- the architecture fix alone didn't fully resolve it. Cell 11 below runs that diagnostic across all 3 folds once trained here.

`rgb_tissue`'s own 3-fold result (already trained, not rerun by this notebook): accuracy 0.8158, macro F1 0.8178, QWK 0.9132 (mean, vs. baseline 0.8168/0.8182/0.9138) -- see `more_experiments/README.md` section 8.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the tissue-fusion dataset

Same `wound-staging-masks-exp-3.zip` as exp-3/exp-4.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-masks-exp-3.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-masks-exp-3"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Fetch the original PIID staging dataset

Needed to train the RGB baseline (step 6) if its checkpoints aren't already on Drive -- same `wound-stage-classification.zip` task_3's own notebook uses.

In [ ]:
ZIP_PATH_STAGE = "/content/drive/MyDrive/thesis/wound-stage-classification.zip"
DATASET_NAME_STAGE = "wound-stage-classification"
DATASET_DIR_STAGE = f"{REPO_DIR}/datasets/{DATASET_NAME_STAGE}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME_STAGE}", recursive=True):
    !unzip -q "{ZIP_PATH_STAGE}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME_STAGE}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME_STAGE}' folder under {EXTRACT_DIR}"
if os.path.islink(DATASET_DIR_STAGE):
    os.remove(DATASET_DIR_STAGE)
os.symlink(candidates[0], DATASET_DIR_STAGE)
!ls -la {DATASET_DIR_STAGE}

## 5. Install dependencies

In [ ]:
!pip install -q -r {REPO_DIR}/task_3/requirements.txt

## 6. RGB baseline checkpoints (fold0/1/2) -- train only the missing ones

This is task_3's OWN, unmodified ablation script, restricted to just the `mit_b1__corn` config -- NOT a different/new RGB model for this experiment, just producing the already-characterized baseline's weights for whichever folds aren't saved yet (fast: ~90s/fold per `task_3/results/task_3_outputs/results/ablation_results.csv`). If you already have all 3 folds on Drive, symlink/copy them into `task_3/results/task_3_outputs/checkpoints/` before this cell and it'll skip training entirely.

## 6. RGB baseline checkpoints (fold0/1/2)

First tries to copy them from `drive/MyDrive/thesis/task_3_outputs/checkpoints/`
(task_3's own Colab notebook output location -- if the 3 `mit_b1__corn_fold{0,1,2}.pt`
files are already there, nothing trains). Only whichever folds are still
missing after that get trained -- task_3's OWN, unmodified ablation
script, restricted to just the `mit_b1__corn` config (not a different/new
RGB model for this experiment, just producing the already-characterized
baseline's weights; fast if needed: ~90s/fold per
`task_3/results/task_3_outputs/results/ablation_results.csv`).

In [ ]:
import glob
import os
import shutil

os.chdir(f"{REPO_DIR}/task_3")
CKPT_DIR = "results/task_3_outputs/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

DRIVE_CKPT_DIR = "/content/drive/MyDrive/thesis/task_3_outputs"
all_drive_pt_files = glob.glob(f"{DRIVE_CKPT_DIR}/**/*.pt", recursive=True)
print(f"{len(all_drive_pt_files)} .pt file(s) found under {DRIVE_CKPT_DIR}")

for fold in range(3):
    local_path = f"{CKPT_DIR}/mit_b1__corn_fold{fold}.pt"
    if os.path.exists(local_path):
        continue
    # match filenames containing "mit_b1", "corn" (not "softmax"), and this fold's index
    candidates = [
        p for p in all_drive_pt_files
        if "mit_b1" in p.lower() and "corn" in p.lower() and "softmax" not in p.lower()
        and f"fold{fold}" in p.lower().replace("_fold", "fold").replace("-fold", "fold")
    ]
    if candidates:
        shutil.copy(candidates[0], local_path)
        print(f"fold{fold}: copied from Drive -> {candidates[0]}")
    else:
        print(f"fold{fold}: no match found under {DRIVE_CKPT_DIR} (listing all .pt names above if this is wrong)")

missing = [f for f in range(3) if not os.path.exists(f"{CKPT_DIR}/mit_b1__corn_fold{f}.pt")]
print("still missing after checking Drive:", missing)

if missing:
    !python check_compatibility.py
    !python run_ablation.py --configs mit_b1__corn --batch 16
else:
    print("all 3 RGB baseline checkpoints present, skipping training")

In [ ]:
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp5_residual_fusion/scripts")
!python test_residual_model.py

## 8. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp5_residual_fusion_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 9. Train Phase 1 — all 3 folds

Each fold loads its OWN matching RGB baseline checkpoint, verifies it reproduces that checkpoint's logits exactly before training, then trains only the tissue CNN + residual head with real early stopping (`patience` from `configs/base.yaml`). Not wrapped in the usual retry loop -- a crash here means investigate (checkpoint mismatch, OOM, etc.), not just restart blindly, per the brief's 'if this verification fails, stop and identify the cause'.

In [ ]:
!python train.py --variant tissue_only --workers 4


## 10. Check results

Compare against the baseline, exp-3, and exp-4 (already recorded in `more_experiments/results/comparison.csv`).

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print("mean test accuracy:", df["accuracy"].mean())
print("mean test macro_f1: ", df["macro_f1"].mean())
print("mean test qwk:      ", df["qwk"].mean())

## 11. Root-cause diagnostics (all folds with a checkpoint)

Per fold: residual-on vs residual-off predictions, gradient norms at convergence, delta_z magnitude vs. CORN decision margins, and the tissue ablation (real vs. zeroed vs. shuffled tissue input) -- the key question: does the correction actually depend on tissue CONTENT, or just on whether tissue is present. Local (fold 1 only) found cosine(real,shuffled)~1.0 and relative L2~0.005 even after the architecture fix -- i.e. still near-content-independent. See if that holds across all 3 folds.

In [ ]:
!python diagnose_residual.py --variant tissue_only
